# Azure ML Workspace에서 모델 다운로드

이 노트북은 Azure ML **Workspace**에 등록된 모델을 다운로드하는 방법을 설명합니다.

## 사전 요구사항
```bash
pip install azure-ai-ml azure-identity mlflow python-dotenv
```

## 환경변수 설정
`.env` 파일을 생성하고 아래 항목을 설정하세요:
```
SUBSCRIPTION_ID=<your-subscription-id>
RESOURCE_GROUP=<your-resource-group>
WORKSPACE_NAME=<your-workspace-name>
MODEL_NAME=<your-model-name>
MODEL_VERSION=<your-model-version>
DOWNLOAD_PATH=./downloaded_model
```

In [ ]:
!pip install azure-ai-ml azure-identity mlflow python-dotenv

## 1. 설정

In [ ]:
import os
from dotenv import load_dotenv
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

NOTEBOOK_DIR = os.path.dirname(__vsc_ipynb_file__)
load_dotenv(os.path.join(NOTEBOOK_DIR, ".env"))

SUBSCRIPTION_ID  = os.getenv("SUBSCRIPTION_ID")
RESOURCE_GROUP   = os.getenv("RESOURCE_GROUP")
WORKSPACE_NAME   = os.getenv("WORKSPACE_NAME")
MODEL_NAME       = os.getenv("MODEL_NAME")
MODEL_VERSION    = os.getenv("MODEL_VERSION")
DOWNLOAD_PATH    = os.path.join(NOTEBOOK_DIR, os.getenv("DOWNLOAD_PATH", "./downloaded_model"))

## 2. MLClient 초기화 (Workspace 연결)

In [ ]:
ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME   # workspace_name 으로 Workspace에 연결
)

print(f"Workspace: {ml_client.workspace_name}")
print(f"Resource Group: {ml_client.resource_group_name}")

## 3. 모델 정보 확인

In [ ]:
model = ml_client.models.get(name=MODEL_NAME, version=MODEL_VERSION)

print(f"모델 이름    : {model.name}")
print(f"버전         : {model.version}")
print(f"타입         : {model.type}")
print(f"Storage URI  : {model.path}")

## 4. 모델 다운로드

In [ ]:
ml_client.models.download(
    name=MODEL_NAME,
    version=MODEL_VERSION,
    download_path=DOWNLOAD_PATH
)

print(f"다운로드 완료: {DOWNLOAD_PATH}")

## 5. 다운로드된 파일 확인

In [ ]:
for root, dirs, files in os.walk(DOWNLOAD_PATH):
    level = root.replace(DOWNLOAD_PATH, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = ' ' * 2 * (level + 1)
    for file in files:
        print(f"{sub_indent}{file}")

## 6. MLflow로 모델 로드 (선택)

In [ ]:
import mlflow

# MLmodel 파일을 찾아서 모델 경로를 동적으로 결정
model_local_path = None
for root, dirs, files in os.walk(DOWNLOAD_PATH):
    if "MLmodel" in files:
        model_local_path = root
        break

if model_local_path is None:
    raise FileNotFoundError(f"MLmodel 파일을 찾을 수 없습니다: {DOWNLOAD_PATH}")

print(f"모델 경로: {model_local_path}")
loaded_model = mlflow.sklearn.load_model(model_local_path)
print(f"모델 로드 완료: {type(loaded_model)}")